# OpenAI Prompts

In [0]:
import asyncio
import os

import aiohttp
import nest_asyncio
import openai
import pandas as pd

from tqdm.notebook import tqdm

In [0]:
INSERT FULL PROFESSORS AND ASSISTANT PROFESSORS DATASETS CELL

In [0]:
openai.api_key = API_KEY

prompt_system = "You are a specialist in Polish academic careers. Your role is to infer scientists' current institution. Specify one parent institution only (in Polish) as output with no faculty, department, centre etc. and additional text. Don't use sentences."

def generate_prompts_prof(row):
    return [
        f"What is the current institution of {row.FIRSTNAME} {row.LASTNAME}?",
        f"What is the current institution of {row.FIRSTNAME} {row.LASTNAME} who is {row.GENDER}?",
        f"What is the current institution of {row.FIRSTNAME} {row.LASTNAME} who was born in {row.BIRTHYEAR}?",
        f"What is the current institution of {row.FIRSTNAME} {row.LASTNAME} who received a full professorship title in {row.DEGREEYEAR}?",
        f"What is the current institution of {row.FIRSTNAME} {row.LASTNAME} who is {row.GENDER} and was born in {row.BIRTHYEAR}?",
        f"What is the current institution of {row.FIRSTNAME} {row.LASTNAME} who is {row.GENDER} and received a full professorship title in {row.DEGREEYEAR}?",
        f"What is the current institution of {row.FIRSTNAME} {row.LASTNAME} who was born in {row.BIRTHYEAR} and received a full professorship title in {row.DEGREEYEAR}?",
        f"What is the current institution of {row.FIRSTNAME} {row.LASTNAME} who is {row.GENDER} and was born in {row.BIRTHYEAR} and received a full professorship title in {row.DEGREEYEAR}?",
    ]

def generate_prompts_dr(row):
    return [
        f"What is the current institution of {row.FIRSTNAME} {row.LASTNAME}?",
        f"What is the current institution of {row.FIRSTNAME} {row.LASTNAME} who is {row.GENDER}?",
        f"What is the current institution of {row.FIRSTNAME} {row.LASTNAME} who was born in {row.BIRTHYEAR}?",
        f"What is the current institution of {row.FIRSTNAME} {row.LASTNAME} who received a doctoral degree in {row.DEGREEYEAR}?",
        f"What is the current institution of {row.FIRSTNAME} {row.LASTNAME} who is {row.GENDER} and was born in {row.BIRTHYEAR}?",
        f"What is the current institution of {row.FIRSTNAME} {row.LASTNAME} who is {row.GENDER} and received a doctoral degree in {row.DEGREEYEAR}?",
        f"What is the current institution of {row.FIRSTNAME} {row.LASTNAME} who was born in {row.BIRTHYEAR} and received a doctoral degree in {row.DEGREEYEAR}?",
        f"What is the current institution of {row.FIRSTNAME} {row.LASTNAME} who is {row.GENDER} and was born in {row.BIRTHYEAR} and received a doctoral degree in {row.DEGREEYEAR}?",
    ]

In [0]:
semaphore = asyncio.Semaphore(100)

async def ask_openai(session: aiohttp.ClientSession, prompt: str) -> str:
    try:
        async with session.post(
            "https://api.openai.com/v1/chat/completions",
            headers={
                "Authorization": f"Bearer {openai.api_key}",
                "Content-Type": "application/json"
            },
            json={
                "model": "gpt-4o",
                "messages": [
                    {"role": "system", "content": prompt_system},
                    {"role": "user", "content": prompt}
                ],
                "temperature": 0,
                "top_p": 1
            },
            timeout=30
        ) as resp:
            data = await resp.json()
            return data["choices"][0]["message"]["content"]
    except Exception as e:
        return f"ERROR: {str(e)}"

async def process_row(row, session: aiohttp.ClientSession, df_set: str):
    prompts = (
        generate_prompts_prof(row) if df_set == "Full Professors"
        else generate_prompts_dr(row)
    )

    async def limited_call(prompt: str) -> str:
        async with semaphore:
            return await ask_openai(session, prompt)

    return await asyncio.gather(*(limited_call(p) for p in prompts))

async def process_dataframe(df, df_set: str):
    async with aiohttp.ClientSession() as session:
        tasks = [
            process_row(row, session, df_set)
            for _, row in df.iterrows()
        ]
        results = []
        for future in tqdm(asyncio.as_completed(tasks), total=len(tasks)):
            res = await future
            results.append(res)
        return results

In [0]:
nest_asyncio.apply()

async def run_processing(dataset: pd.DataFrame, label: str):
    results = await process_dataframe(dataset, label)
    for i in range(8):
        dataset[f"AI_Prompt_{i+1}"] = [r[i] for r in results]
    return spark.createDataFrame(dataset)

RESULTS_FULL_PROFESSORS = await run_processing(DATASET_FULL_PROFESSORS, "Full Professors")
RESULTS_ASSISTANT_PROFESSORS = await run_processing(DATASET_ASSISTANT_PROFESSORS, "Assistant Professors")

In [0]:
SAVE FULL PROFESSORS AND ASSISTANT PROFESSORS CHATGPT RESULTS CELL

# Classification metrics

In [0]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score

In [0]:
INSERT FULL PROFESSORS AND ASSISTANT PROFESSORS CHATGPT RESULTS CELL

In [0]:
def compute_metrics(y_true, y_pred):
    accuracy = round(accuracy_score(y_true, y_pred), 4)
    precision = round(precision_score(y_true, y_pred, average='weighted', zero_division=0), 4)
    recall = round(recall_score(y_true, y_pred, average='weighted', zero_division=0), 4)

    return accuracy, precision, recall

In [0]:
def calculate_classification_metrics(dataframes, dataset_names):
    results = []

    for df, dataset_name in zip(dataframes, dataset_names):
        pred_cols = df.columns[-8:]
        n_total = len(df)

        for col in pred_cols:
            valid = df[['INSTITUTION', col]].dropna().copy()
            n_valid = len(valid)

            if n_valid == 0:
                continue

            y_true = valid['INSTITUTION'].str.lower()
            y_pred = valid[col].str.lower()

            try:
                metrics = compute_metrics(y_true, y_pred)

                results.append([dataset_name, col, *metrics])
            except Exception as e:
                print(f"Error in dataset '{dataset_name}', column '{col}': {e}")

    columns = ['Dataset', 'Parameters', 'Accuracy', 'Precision', 'Recall']
    
    return pd.DataFrame(results, columns=columns) if results else pd.DataFrame(columns=columns)

metrics = calculate_classification_metrics([RESULTS_FULL_PROFESSORS, RESULTS_ASSISTANT_PROFESSORS], ['Full Professors', 'Assistant Professors'])

In [0]:
display(metrics)

### By Institution type

In [0]:
def calculate_classification_metrics(dataframes, dataset_names):
    results = []

    for df, dataset_name in zip(dataframes, dataset_names):
        prediction_cols = df.columns[-8:]
        group_col = df.columns[6]

        for group_name, group_df in df.groupby(group_col):
            n_total = len(group_df)

            for col in prediction_cols:
                valid = group_df[['INSTITUTION', col]].dropna().copy()
                n_valid = len(valid)

                if n_valid == 0:
                    continue

                y_true = valid['INSTITUTION'].str.lower()
                y_pred = valid[col].str.lower()

                try:
                    metrics = compute_metrics(y_true, y_pred)

                    results.append([group_name, dataset_name, col, *metrics])
                except Exception as e:
                    print(f"Error in group '{group_name}', dataset '{dataset_name}', column '{col}': {e}")

    columns = ['Uni Group', 'Dataset', 'Parameters', 'Accuracy', 'Precision', 'Recall']
    
    return pd.DataFrame(results, columns=columns) if results else pd.DataFrame(columns=columns)

metrics = calculate_classification_metrics([RESULTS_FULL_PROFESSORS, RESULTS_ASSISTANT_PROFESSORS], ['Full Professors', 'Assistant Professors'])

In [0]:
display(metrics)